In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
class ToyModel(nn.Module):
    def __init__(self, n_features: int, n_hidden: int):
        super().__init__()
        # W has shape (n_hidden, n_features). Each column is the 'direction' the model uses
        # to represent that feature in hidden space.
        self.W = nn.Parameter(torch.empty(n_hidden, n_features))
        nn.init.xavier_normal_(self.W)
        self.b = nn.Parameter(torch.zeros(n_features))

    def forward(self, x):
        # x: (batch, n_features)
        hidden = x @ self.W.T          # (batch, n_hidden) -- encode
        out = hidden @ self.W + self.b # (batch, n_features) -- decode (tied weights)
        return F.relu(out)

In [ ]:
def generate_batch(batch_size: int, n_features: int, sparsity: float, device):
    """Returns a (batch_size, n_features) tensor of sparse non-negative features."""
    values = torch.rand(batch_size, n_features, device=device)
    mask   = torch.rand(batch_size, n_features, device=device) > sparsity
    return values * mask

# sanity check: at S=0.9, roughly 10% of entries should be nonzero
sample = generate_batch(1000, n_features=5, sparsity=0.9, device=device)
print(f'Fraction of nonzero entries: {(sample > 0).float().mean().item():.3f} (expected ~0.10)')

In [ ]:
def make_importance(n_features: int, decay: float = 0.7, device='cpu'):
    return decay ** torch.arange(n_features, device=device, dtype=torch.float32)

def loss_fn(out, target, importance):
    # out, target: (batch, n_features); importance: (n_features,)
    return ((out - target) ** 2 * importance).mean()

print('Importance weights (n=5):', make_importance(5).numpy())

In [ ]:
def train(n_features=5, n_hidden=2, sparsity=0.0, n_steps=10_000, batch_size=1024,
          lr=1e-3, verbose=False):
    model = ToyModel(n_features=n_features, n_hidden=n_hidden).to(device)
    importance = make_importance(n_features, device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps)

    losses = []
    for step in range(n_steps):
        x = generate_batch(batch_size, n_features, sparsity, device=device)
        out = model(x)
        loss = loss_fn(out, x, importance)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())
        if verbose and step % 1000 == 0:
            print(f'step {step:5d}  loss {loss.item():.5f}')

    return model, losses

In [ ]:
model, losses = train(sparsity=0.9, verbose=True)

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss (log scale)')
plt.title('Training loss (sparsity=0.9)')
plt.grid(True, alpha=0.3)
plt.show()